In [ ]:
from pathlib import Path
import sys
import base64
import requests
import pandas as pd

sys.path.append("/home/jonas/PycharmProjects/tikzcodegenerator")

from evaluation.promptfoo.pf_utils.tikz_rendering import render_tex_to_png


OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "qwen3.6:35b-a3b-q4_K_M"

code_folder = Path("/home/jonas/Datasets/TikZ/train-big/references")
image_folder = Path("/home/jonas/Datasets/TikZ/train-big/images")
description_folder = Path("/home/jonas/Datasets/TikZ/train-big/vlm_descriptions")
manifest_path = Path("/home/jonas/Datasets/TikZ/train-big/manifest.csv")

image_folder.mkdir(parents=True, exist_ok=True)
description_folder.mkdir(parents=True, exist_ok=True)

txt_files = sorted(code_folder.glob("*.txt"))


def image_to_base64(image_path: Path) -> str:
    return base64.b64encode(image_path.read_bytes()).decode("utf-8")


def describe_image_with_ollama(image_path: Path) -> str:
    prompt = (
        "Describe this image precisely and concisely. "
        "Focus on visible geometric elements, text labels, colors, "
        "arrows, lines, nodes, axes, and their spatial arrangement. "
        "The description should be suitable for reconstructing TikZ code from it later. "
        "Respond only with the description."
    )

    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "images": [image_to_base64(image_path)],
        "stream": False,
    }

    response = requests.post(OLLAMA_URL, json=payload, timeout=300)
    response.raise_for_status()

    return response.json()["response"].strip()


manifest_rows = []

for file in txt_files:
    print(file)

    tex_code = file.read_text(encoding="utf-8")

    generated_image = image_folder / f"{file.stem}.png"
    description_path = description_folder / f"{file.stem}.txt"

    render_tex_to_png(
        tex_code=tex_code,
        output_path=generated_image,
    )

    description = describe_image_with_ollama(generated_image)
    description_path.write_text(description, encoding="utf-8")

    manifest_rows.append({
        "code_path": str(file),
        "image_path": str(generated_image),
        "vlm_description_path": str(description_path),
    })

df = pd.DataFrame(manifest_rows)
df.to_csv(manifest_path, index=False)

print(f"Manifest written to: {manifest_path}")
print(f"Descriptions written to: {description_folder}")

/home/jonas/Datasets/TikZ/train-big/references/datikz-test_row_00000000_without_text.txt
/home/jonas/Datasets/TikZ/train-big/references/datikz-test_row_00000001_without_text.txt
/home/jonas/Datasets/TikZ/train-big/references/datikz-test_row_00000002_without_text.txt
/home/jonas/Datasets/TikZ/train-big/references/datikz-test_row_00000003_without_text.txt
/home/jonas/Datasets/TikZ/train-big/references/datikz-test_row_00000004_without_text.txt
/home/jonas/Datasets/TikZ/train-big/references/datikz-test_row_00000005_without_text.txt
/home/jonas/Datasets/TikZ/train-big/references/datikz-test_row_00000006_without_text.txt
/home/jonas/Datasets/TikZ/train-big/references/datikz-test_row_00000007_without_text.txt
/home/jonas/Datasets/TikZ/train-big/references/datikz-test_row_00000008_without_text.txt
/home/jonas/Datasets/TikZ/train-big/references/datikz-test_row_00000009_without_text.txt
/home/jonas/Datasets/TikZ/train-big/references/datikz-test_row_00000010_without_text.txt
Manifest written to: 